# Task 2: Laning & Overtaking with SB3 PPO

This notebook trains a PPO agent on the newer `highway-env` / Stable-Baselines3 API, following the older Task 2 setup from `racetrack-agents` where three slower non-agent vehicles are spawned and the ego vehicle must lane-follow while overtaking.

The older DQN command used `--spawn_vehicles 3`, `--batch_size 256`, `--lr 0.00005`, `--lr_decay`, `--arch Identity`, and `--fc_layers 3`. The cells below map those ideas to SB3 PPO with a 3-layer MLP policy, linear learning-rate decay, and `other_vehicles=3` in the `racetrack-oval-v0` config.

In [1]:
# If this notebook is running in a fresh environment, install the core packages first.
# In the local repo environment you can usually leave this cell commented out.
#
# %pip install "highway-env>=1.8" "stable-baselines3[extra]>=2.0" tensorboard moviepy

from pathlib import Path
from copy import deepcopy
import base64
import os
import platform
import random
import subprocess
import sys
import time

import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import highway_env  # Registers highway-env environments in many versions.
import numpy as np
import torch

from IPython.display import HTML, display
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback, EvalCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv

# Newer gymnasium versions can register an external environment package explicitly.
# Older highway-env versions register on import, so we keep this tolerant.
try:
    gym.register_envs(highway_env)
except Exception:
    pass


c:\Users\16469\anaconda3\envs\circuit\lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


## Experiment Config

Task 2 is represented by `other_vehicles=3`. The notebook now starts from `RacetrackEnvOval.default_config()` and overrides only the pieces that define this experiment, so future highway-env API changes are easier to absorb.

In [2]:
SEED = 42
ENV_ID = "racetrack-v0"

# Keep full training as the default. For an end-to-end notebook smoke test, run
# `FAST_DEV_RUN=1` in the process environment before executing the notebook.
FAST_DEV_RUN = True

USE_RICH_OCCUPANCY_FEATURES = True
USE_THROTTLE = True
USE_SUBPROC = False

# Prefer CPU in notebooks unless a CUDA-enabled GPU is available and stable.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

EXP_ID = "sb3_ppo_speed_config"
if FAST_DEV_RUN:
    EXP_ID += "_fastdev"

WORK_DIR = Path.cwd()

RUN_DIR = WORK_DIR / "runs" / EXP_ID
MODEL_DIR = RUN_DIR / "models"
BEST_MODEL_DIR = MODEL_DIR / "best"
CHECKPOINT_DIR = MODEL_DIR / "checkpoints"
LOG_DIR = RUN_DIR / "logs"
VIDEO_DIR = RUN_DIR / "videos"
TB_LOG_DIR = RUN_DIR / "tensorboard"

for directory in [MODEL_DIR, BEST_MODEL_DIR, CHECKPOINT_DIR, LOG_DIR, VIDEO_DIR, TB_LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Reproducibility: exact runs can still vary across machines/GPU kernels.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

Using device: cuda


In [3]:
from highway_env.envs import racetrack_env
from race_env import RacetrackFast
racetrack_env.RacetrackFast = RacetrackFast
gym.register(id=ENV_ID, entry_point="race_env:RacetrackFast")

c:\Users\16469\anaconda3\envs\circuit\lib\site-packages\gymnasium\envs\registration.py:694: UserWarning: WARN: Overriding environment racetrack-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


In [4]:
# Fast mode now performs a real short PPO run with multiple rollouts/evaluations so progress is visible.
FAST_DEV_TIMESTEPS = int(os.environ.get("FAST_DEV_TIMESTEPS", "8192"))

# Full mode keeps the older command's 5000-episode intent, using the current API's episode horizon.
N_EPISODES = 5000
TOTAL_TIMESTEPS = FAST_DEV_TIMESTEPS if FAST_DEV_RUN else N_EPISODES * RacetrackFast.default_config["duration"]

# Use one env in notebooks on Windows; full mode can use several vectorized workers.
N_ENVS = 1 if FAST_DEV_RUN else min(8, max(1, os.cpu_count() or 1))

# PPO minibatch size: smaller in fast mode, older command value in full mode.
BATCH_SIZE = 128 if FAST_DEV_RUN else 256

# Rollout length per env before each PPO update; longer rollouts stabilize PPO on this task.
N_STEPS = 1024 if FAST_DEV_RUN else 2048

# Starting learning rate, mapped from the older `--lr 0.00005` setting but slightly higher for quicker learning.
LEARNING_RATE = 2.5e-4 #5e-5

# Number of SGD passes per PPO update; fewer in fast mode keeps iteration time reasonable.
N_EPOCHS = 6 if FAST_DEV_RUN else 10

# Evaluation cadence in environment steps; fast mode evaluates often so you can see progress.
EVAL_FREQ = 2048 if FAST_DEV_RUN else max(10_000 // N_ENVS, 1)

# Evaluation episodes per callback; three is enough to see trend during a quick run.
N_EVAL_EPISODES = 3 if FAST_DEV_RUN else 5

# A small entropy bonus is helpful with richer occupancy features and throttle control.
ENT_COEF = 0.001 if USE_RICH_OCCUPANCY_FEATURES else 0.0005

# Slightly smaller clip range helps learning stay stable when the action space grows.
CLIP_RANGE = 0.12

# Stronger discounting and GAE smoothing can improve horizon-aware lane-following and overtaking behavior.
GAMMA = 0.99
GAE_LAMBDA = 0.97
MAX_GRAD_NORM = 0.8


## Build Training and Evaluation Environments

SB3 trains on vectorized environments. `DummyVecEnv` is the safest default inside notebooks on Windows. For longer command-line runs, set `USE_SUBPROC = True`.

In [5]:
def make_task2_env(render_mode=None):
    """Create one Task 2 racetrack environment."""
    return gym.make(ENV_ID, config=RacetrackFast.default_config(), render_mode=render_mode)

vec_env_cls = SubprocVecEnv if USE_SUBPROC and N_ENVS > 1 else DummyVecEnv

train_env = make_vec_env(
    lambda: make_task2_env(),
    n_envs=N_ENVS,
    seed=SEED,
    vec_env_cls=vec_env_cls,
)
# Evaluation stays single-env so callback results are easy to interpret.
eval_env = make_vec_env(
    lambda: make_task2_env(),
    n_envs=1,
    seed=SEED + 10_000,
    vec_env_cls=DummyVecEnv,
)


## Define PPO

The PPO policy uses a 3-layer actor and critic MLP, matching the spirit of `--arch Identity --fc_layers 3` from the older code: flatten the occupancy grid, then learn dense policy/value heads. Comments below explain every model setting that differs from SB3 defaults or maps to the older command.

In [6]:
def linear_schedule(initial_value):
    """SB3 schedule: progress_remaining moves from 1.0 to 0.0 during training."""
    def schedule(progress_remaining):
        return progress_remaining * initial_value
    return schedule

policy_kwargs = {
    # Tanh matches the older TensorFlow PPO hidden-layer style and works well with normalized features.
    "activation_fn": torch.nn.Tanh,

    # Three 256-unit layers map the old `--fc_layers 3` / default `--fc_width 256` idea to SB3.
    "net_arch": {
        "pi": [256, 256, 256],  # Actor network: outputs the lateral continuous-control distribution.
        "vf": [256, 256, 256],  # Critic network: estimates state value for PPO advantage learning.
    },
}

model = PPO(
    policy="MlpPolicy",  # Flattened occupancy-grid input, equivalent in spirit to the older Identity backbone.
    env=train_env,  # Vectorized Task 2 racetrack environment.
    learning_rate=linear_schedule(LEARNING_RATE),  # Implements the older `--lr_decay` behavior.
    n_steps=N_STEPS,  # Rollout length before each PPO update.
    batch_size=BATCH_SIZE,  # Minibatch size for PPO optimization.
    n_epochs=N_EPOCHS,  # Number of optimization passes over each rollout buffer.
    gamma=GAMMA,  # Discount factor tuned for longer-horizon lane-following and overtaking.
    gae_lambda=GAE_LAMBDA,  # GAE smoothing tuned for richer rewards.
    clip_range=CLIP_RANGE,  # Slightly smaller clip range improves stability with richer observations.
    ent_coef=ENT_COEF,  # Small entropy bonus to keep exploration alive during overtaking.
    max_grad_norm=MAX_GRAD_NORM,  # Gradient clipping for stable policy updates.
    policy_kwargs=policy_kwargs,  # Actor/critic architecture defined above.
    tensorboard_log=str(TB_LOG_DIR),  # Training curves and eval metrics.
    seed=SEED,  # Reproducible initialization and rollout seeds where supported.
    verbose=1,  # Print rollout/evaluation progress in notebook output.
    device=DEVICE,
)

model.policy


Using cuda device


ActorCriticPolicy(
  (features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (pi_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (vf_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (mlp_extractor): MlpExtractor(
    (policy_net): Sequential(
      (0): Linear(in_features=1584, out_features=256, bias=True)
      (1): Tanh()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): Tanh()
      (4): Linear(in_features=256, out_features=256, bias=True)
      (5): Tanh()
    )
    (value_net): Sequential(
      (0): Linear(in_features=1584, out_features=256, bias=True)
      (1): Tanh()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): Tanh()
      (4): Linear(in_features=256, out_features=256, bias=True)
      (5): Tanh()
    )
  )
  (action_net): Linear(in_features=256, out_features=2, bias=True)
  (value_net): Linear(in

## Train and Save the Best Model

`EvalCallback` periodically runs deterministic evaluations and writes the best model to disk. TensorBoard logs are stored under `runs/sb3_ppo_task2_laning_overtaking/logs`.

In [7]:
# Start TensorBoard in the notebook while training is running.
# This opens the log directory in a background server and prints the local URL.
#
%load_ext tensorboard
%tensorboard --logdir TB_LOG_DIR

# If the magic above is not available, you can also launch a standalone server from a terminal:
# tensorboard --logdir "runs/sb3_ppo_laning_overtaking_richocc_throttle_fastdev/tensorboard" --host 127.0.0.1 --port 6006


Reusing TensorBoard on port 6006 (pid 59216), started 1 day, 0:56:29 ago. (Use '!kill 59216' to kill it.)

In [8]:
class EarlyStoppingCallback(BaseCallback):
    def __init__(self, check_freq=10000, patience=3, min_delta=0.0, verbose=0):
        super().__init__(verbose)
        self.check_freq = check_freq
        self.patience = patience
        self.min_delta = min_delta
        self.best_mean_reward = -np.inf
        self.epochs_without_improvement = 0

    def _on_step(self) -> bool:
        if self.n_calls % self.check_freq == 0:
            # Use the current training reward estimate from the rollout buffer.
            # This is intentionally conservative and does not require a full eval loop.
            current_mean_reward = np.mean(self.locals.get("rewards", [0.0]))
            if current_mean_reward > self.best_mean_reward + self.min_delta:
                self.best_mean_reward = current_mean_reward
                self.epochs_without_improvement = 0
            else:
                self.epochs_without_improvement += 1
            if self.epochs_without_improvement >= self.patience:
                if self.verbose:
                    print(f"Early stopping at step {self.num_timesteps} with no improvement.")
                return False
        return True


early_stop_callback = EarlyStoppingCallback(check_freq=max(5_000 // N_ENVS, 1), patience=2, min_delta=0.1, verbose=1)

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=str(BEST_MODEL_DIR),
    log_path=str(LOG_DIR / "eval"),
    eval_freq=EVAL_FREQ,
    n_eval_episodes=N_EVAL_EPISODES,
    deterministic=True,
    render=False,
)

checkpoint_callback = CheckpointCallback(
    save_freq=max(50_000 // N_ENVS, 1),
    save_path=str(CHECKPOINT_DIR),
    name_prefix="ppo_task2_checkpoint",
)

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=[early_stop_callback, eval_callback, checkpoint_callback],
    tb_log_name="PPO_task2_tuned",
)

model.save(MODEL_DIR / "ppo_task2_last")
train_env.close()
eval_env.close()


Logging to c:\Users\16469\Desktop\circuit\racetrack-agents\runs\sb3_ppo_speed_config_fastdev\tensorboard\PPO_task2_tuned_2
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 10.4     |
|    ep_rew_mean     | 7.06     |
| time/              |          |
|    fps             | 47       |
|    iterations      | 1        |
|    time_elapsed    | 21       |
|    total_timesteps | 1024     |
---------------------------------
Eval num_timesteps=2048, episode_reward=32.11 +/- 3.11
Episode length: 36.33 +/- 2.49
-----------------------------------------
| eval/                   |             |
|    mean_ep_length       | 36.3        |
|    mean_reward          | 32.1        |
| time/                   |             |
|    total_timesteps      | 2048        |
| train/                  |             |
|    approx_kl            | 0.014204515 |
|    clip_fraction        | 0.233       |
|    clip_range           | 0.12        |
|    entropy_loss         | -2.

In [9]:
import importlib
import os
import numpy as np

for module_name in ["onnx", "onnxruntime"]:
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        os.system('pip install --quiet "onnx==1.12.0" "onnxruntime==1.12.0"')

import onnx
import onnxruntime as ort


class ActorOnlyPolicyWrapper(torch.nn.Module):
    """Export the trained policy as a deterministic actor that returns the mean action."""
    def __init__(self, policy):
        super().__init__()
        self.policy = policy

    def forward(self, obs):
        obs = obs.float()
        if obs.dim() == 1:
            obs = obs.unsqueeze(0)
        features = self.policy.features_extractor(obs)
        latent_pi, _ = self.policy.mlp_extractor(features)
        mean_actions = self.policy.action_net(latent_pi)
        return mean_actions


best_model_path = BEST_MODEL_DIR / "best_model.zip"
last_model_path = MODEL_DIR / "ppo_task2_last.zip"

if best_model_path.exists():
    trained_model = PPO.load(best_model_path)
else:
    trained_model = PPO.load(last_model_path)

onnx_path = MODEL_DIR / "ppo_task2_tuned.onnx"
wrapper = ActorOnlyPolicyWrapper(trained_model.policy).to("cpu")
wrapper.eval()

obs_dim = int(np.prod(train_env.observation_space.shape))
dummy_obs = torch.randn(1, obs_dim, dtype=torch.float32)

torch.onnx.export(
    wrapper,
    dummy_obs,
    str(onnx_path),
    export_params=True,
    opset_version=12,
    do_constant_folding=True,
    input_names=["obs"],
    output_names=["action_mean"],
    dynamic_axes={"obs": {0: "batch_size"}, "action_mean": {0: "batch_size"}},
    dynamo=False,
)

onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)

session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
example_obs = np.random.randn(1, obs_dim).astype(np.float32)
action_mean = session.run(None, {"obs": example_obs})[0]

print(f"Exported ONNX model to {onnx_path}")
print(f"ONNX opset version: {onnx_model.opset_import[0].version}")
print(f"Sample output shape: {action_mean.shape}")


Exported ONNX model to c:\Users\16469\Desktop\circuit\racetrack-agents\runs\sb3_ppo_speed_config_fastdev\models\ppo_task2_tuned.onnx
ONNX opset version: 12
Sample output shape: (1, 2)


C:\Users\16469\AppData\Local\Temp\ipykernel_69852\99746512.py:46: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


## Load the Best PPO Checkpoint

If training was interrupted before an evaluation improved, fall back to the last saved model.

In [10]:
best_model_path = BEST_MODEL_DIR / "best_model.zip"
last_model_path = MODEL_DIR / "ppo_task2_last.zip"

if best_model_path.exists():
    trained_model = PPO.load(best_model_path)
    print(f"Loaded best model: {best_model_path}")
else:
    trained_model = PPO.load(last_model_path)
    print(f"Best model was not found, loaded last model: {last_model_path}")


Loaded best model: c:\Users\16469\Desktop\circuit\racetrack-agents\runs\sb3_ppo_speed_config_fastdev\models\best\best_model.zip


## Find and Export the Best Evaluation Episode

The first pass evaluates deterministic rollouts over fixed seeds without recording. The best seed is then replayed once with `RecordVideo`, producing a single video for the strongest episode found in this sweep.

In [11]:
def run_episode(model, seed, record=False, name_prefix="ppo_task2_best_episode"):
    """Run one deterministic episode and optionally record it to VIDEO_DIR."""
    render_mode = "rgb_array" if record else None
    env = gym.make(ENV_ID, config=RacetrackFast.default_config(), render_mode=render_mode)

    if record:
        env = RecordVideo(
            env,
            video_folder=str(VIDEO_DIR),
            name_prefix=name_prefix,
            episode_trigger=lambda episode_id: episode_id == 0,
        )
    obs, info = env.reset(seed=seed)
    done = False
    truncated = False
    total_reward = 0.0
    episode_length = 0

    while not (done or truncated):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env.step(action)
        total_reward += float(reward)
        episode_length += 1
        if record:
            env.render()

    env.close()
    return total_reward, episode_length

candidate_seeds = list(range(SEED, SEED + (1 if FAST_DEV_RUN else 25)))
episode_scores = []

for seed in candidate_seeds:
    reward, length = run_episode(trained_model, seed=seed, record=False)
    episode_scores.append({"seed": seed, "reward": reward, "length": length})

best_episode = max(episode_scores, key=lambda item: item["reward"])
best_episode


{'seed': 42, 'reward': 1072.9339986376967, 'length': 1101}

In [12]:
video_prefix = f"ppo_task2_best_seed_{best_episode['seed']}"
recorded_reward, recorded_length = run_episode(
    trained_model,
    seed=best_episode["seed"],
    record=True,
    name_prefix=video_prefix,
)

video_files = sorted(VIDEO_DIR.glob(f"{video_prefix}*.mp4"), key=lambda path: path.stat().st_mtime)
best_video_path = video_files[-1] if video_files else None

print(f"Recorded reward: {recorded_reward:.3f}")
print(f"Recorded length: {recorded_length}")
print(f"Video path: {best_video_path}")


c:\Users\16469\anaconda3\envs\circuit\lib\site-packages\gymnasium\wrappers\record_video.py:94: UserWarning: WARN: Overwriting existing videos at c:\Users\16469\Desktop\circuit\racetrack-agents\runs\sb3_ppo_speed_config_fastdev\videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


MoviePy - Building video c:\Users\16469\Desktop\circuit\racetrack-agents\runs\sb3_ppo_speed_config_fastdev\videos\ppo_task2_best_seed_42-episode-0.mp4.
MoviePy - Writing video c:\Users\16469\Desktop\circuit\racetrack-agents\runs\sb3_ppo_speed_config_fastdev\videos\ppo_task2_best_seed_42-episode-0.mp4



MoviePy - Done !
MoviePy - video ready c:\Users\16469\Desktop\circuit\racetrack-agents\runs\sb3_ppo_speed_config_fastdev\videos\ppo_task2_best_seed_42-episode-0.mp4
Recorded reward: 1072.934
Recorded length: 1101
Video path: c:\Users\16469\Desktop\circuit\racetrack-agents\runs\sb3_ppo_speed_config_fastdev\videos\ppo_task2_best_seed_42-episode-0.mp4


## Display the Exported Video

In [13]:
def show_video(video_path, width=720):
    """Embed an exported mp4 directly in the notebook."""
    video_path = Path(video_path)
    video_bytes = video_path.read_bytes()
    encoded = base64.b64encode(video_bytes).decode("ascii")
    display(HTML(f"""
    <video width="{width}" controls>
      <source src="data:video/mp4;base64,{encoded}" type="video/mp4">
    </video>
    """))

if best_video_path is not None:
    show_video(best_video_path)
else:
    print("No video file was found. Check that moviepy/ffmpeg are installed and rerun the recording cell.")


## Optional: TensorBoard

Run this cell while training or after training to inspect reward, loss, entropy, KL, and evaluation curves.

In [14]:
# Uncomment these lines in an interactive notebook session.

